# Quantonium Video — Colab training notebook and journal
**Prepared 29 October 2026.** Resume the existing V0.8 interpolator on a Colab GPU, save verified snapshots to Drive, evaluate, and render JC.

This notebook trains your **87,817-parameter RGBA interpolator**, with frozen RAFT motion guidance and JC rehearsal. For **Wan generator training**, open the [dedicated Wan LoRA notebook](https://colab.research.google.com/github/Anharmoniclabs/quantoniumoslinux/blob/feat/quantonium-anime-video-v01/runtime/experiments/quantonium_video/Quantonium_Wan_LoRA_Colab.ipynb). Keep this interpolator stopped while Wan uses the GPU; each model has its own resumable weights.

### Start here
1. Use a GPU Colab runtime (L4 if available; the interpolator can use other CUDA GPUs). Use a separate available session; do not interrupt your PhiState language-model training to run this notebook.
2. The checkpoint bundle for this handoff is already staged in `MyDrive/Quantonium/Video/OpenV08/QVideo_Open_Weights_Data_Checkpoint.zip`. Mount the same Google Drive account. For another account, copy the bundle there or use the upload cell. Git contains code, not weights.
3. Run setup, restore, and start cells. Default: **continuous training from the latest saved phase**, with verified Drive saves every 25 updates.
4. Leave the Start cell running to see live progress. Press its stop button once to request a clean save and wait for it to exit. Later cells run after training stops.

Original source: [GitHub](https://github.com/Anharmoniclabs/quantoniumoslinux/tree/feat/quantonium-anime-video-v01/runtime/experiments/quantonium_video).
The model is still experimental. A completed step target is not an animation-quality approval.


In [ ]:
# 1. Configuration and GPU check. Does not reinstall or replace Colab's CUDA PyTorch.
from pathlib import Path
import os, sys, json, subprocess, shutil, time, uuid, hashlib, zipfile
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before training this notebook."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
CODE_COMMIT = "afb8c0c2cbabc5f0e61ef90c0044410bdb7ed563"
TARGET_PHASE_STEP = 0  # 0 = continuous until you stop or the runtime ends.
SAVE_EVERY = 25
DRIVE_ROOT = Path("/content/drive/MyDrive/Quantonium/Video/OpenV08")
BUNDLE_NAME = "QVideo_Open_Weights_Data_Checkpoint.zip"
BUNDLE_SHA256 = "558c4b70f22e7142fee0d99e0f09bd938b2c37af627f2186790c37eb466da641"
REPO = Path("/content/quantonium_video_continuous_source")
LOCAL = Path("/content/quantonium_open_video")
def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()



## 2. Mount Drive and load the pinned code
Only the dedicated `Quantonium/Video/OpenV08` folder is used. PhiState folders and the original V0.7 automation are untouched. Colab resources can disconnect; snapshots preserve files, not a running process. [Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pillow"], check=True)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
if not (REPO/".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "remote", "add", "origin",
                    "https://github.com/Anharmoniclabs/quantoniumoslinux.git"], check=True)
    subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set",
                    "runtime/experiments/quantonium_video"], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth=1", "origin", CODE_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == CODE_COMMIT, "Code checkout differs. Use a separate clean source directory."
LAB = REPO/"runtime/experiments/quantonium_video"
sys.path.insert(0, str(LAB))
sys.modules.pop("colab_video_runtime", None)  # Refresh helper when upgrading this notebook.
from colab_video_runtime import save_snapshot, restore_latest
from train_open_v08 import OpenData, VERSION
from train_v07 import PoseData, PoseInterpolator, DEFAULT_ASSETS
# This includes a real CUDA forward/backward/evaluation test when a GPU is present.
subprocess.run([sys.executable, "-m", "unittest", "test_v07", "test_open_video",
                "test_colab_video", "-v"], cwd=LAB, check=True)
CUDA_TEST_PASSED = True



## 3. Restore the saved weights and reviewed data
The bundle contains the immutable V0.7 parent at step **1,750**, V0.8 latest at phase **100**, best at phase **75**, 57 training triplets, 49 validation triplets, the original Blender shot, and RAFT weights.

A later valid Drive snapshot takes priority. Restoration refuses different data/configuration or a rollback over a newer local checkpoint. The bundle checksum is pinned; missing or corrupt inputs never trigger fresh training.


In [ ]:
BUNDLE = DRIVE_ROOT/BUNDLE_NAME
if not BUNDLE.exists():
    from google.colab import files
    print("Choose", BUNDLE_NAME)
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded:
        raise FileNotFoundError("The required checkpoint bundle was not selected.")
    upload_path = Path("/content")/BUNDLE_NAME
    upload_path.write_bytes(uploaded[BUNDLE_NAME])
    assert digest(upload_path) == BUNDLE_SHA256, "Uploaded bundle checksum differs."
    shutil.copyfile(upload_path, BUNDLE)
    del uploaded
assert digest(BUNDLE) == BUNDLE_SHA256, "Stored bundle checksum differs."
if not LOCAL.exists():
    staging = Path("/content")/("qvideo_restore_"+uuid.uuid4().hex)
    staging.mkdir()
    with zipfile.ZipFile(BUNDLE) as z:
        assert sum(i.file_size for i in z.infolist()) < 512*1024*1024
        for item in z.infolist():
            target = (staging/item.filename).resolve()
            assert target.is_relative_to(staging.resolve()), "Unsafe archive path"
            assert item.filename.startswith("qvideo_open/"), "Unexpected archive member"
        z.extractall(staging)
    (staging/"qvideo_open").rename(LOCAL)
    shutil.rmtree(staging)
RUN = LOCAL/"run"
DATA = LOCAL/"data"
SHOT = LOCAL/"blender_shot"
PARENT = LOCAL/"parent.pt"
STOP_FILE = LOCAL/"STOP_REQUEST"
def worker_active():
    import fcntl
    with (RUN/"training.lock").open("a") as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            return True
        return False
assert not worker_active(), "A trainer is active; do not restore over it."
assert digest(PARENT) == "e211a2ea5ecae42931a152d38aca59758bbc49b1db2640c942dd8ff612fb3c24"
state = torch.load(RUN/"latest.pt", map_location="cpu", weights_only=True)
assert state["config"]["version"] == VERSION
restored = restore_latest(DRIVE_ROOT, RUN, state["config"])
if restored is None:
    restored = save_snapshot(RUN, DRIVE_ROOT)  # Preserve starting phase100 before GPU work.
state = torch.load(RUN/"latest.pt", map_location="cpu", weights_only=True)
data = OpenData(DATA, state["config"]["size"])
assert data.fingerprint == state["config"]["data"]
jc = PoseData(DEFAULT_ASSETS, state["config"]["size"])
jc.attach_shot(SHOT)
assert jc.fingerprint == state["config"]["jc"] and jc.shot_hash == state["config"]["shot"]
print("Verified phase step:", state["step"], "| parent:", state["config"]["parent_step"])
print("Train / validation:", len(data.items["train"]), "/", len(data.items["validation"]))
print("Drive snapshot:", restored)
print("Target:", "continuous" if TARGET_PHASE_STEP == 0 else TARGET_PHASE_STEP)



## 4. Start or resume GPU training
Batch size, learning rate and rehearsal weight come from the saved checkpoint. Changing these would break exact resume configuration.

Every 25 updates, training evaluates both splits, saves locally, then writes an immutable Drive snapshot and reads it back to verify hashes. It prints **DRIVE SNAPSHOT VERIFIED** only after verification. If saving fails, the trainer exits instead of continuing without backups.

The original optimizer and weights resume. CPU-to-CUDA results are not bitwise identical because kernels differ. No automatic checkpoint promotion occurs.

Default `TARGET_PHASE_STEP = 0` removes the step cap. This cell stays running and displays progress; it does not restart a disconnected Colab runtime.


In [ ]:
assert CUDA_TEST_PASSED and torch.cuda.is_available()
assert not worker_active(), "Training is already running."
state = torch.load(RUN/"latest.pt", map_location="cpu", weights_only=True)
cfg = state["config"]
if TARGET_PHASE_STEP > 0 and state["step"] >= TARGET_PHASE_STEP:
    print("Target already reached. Raise TARGET_PHASE_STEP only when you want another training phase.")
else:
    STOP_FILE.unlink(missing_ok=True)
    command = [sys.executable, "-u", str(LAB/"train_open_v08.py"),
        "--init", str(PARENT), "--out", str(RUN), "--data", str(DATA), "--shot", str(SHOT),
        "--require-resume", "--steps", str(TARGET_PHASE_STEP), "--save-every", str(SAVE_EVERY),
        "--batch", str(cfg["batch"]), "--lr", str(cfg["lr"]), "--jc-weight", str(cfg["jc_weight"]),
        "--device", "cuda", "--threads", "4", "--backup-dir", str(DRIVE_ROOT),
        "--stop-file", str(STOP_FILE)]
    log_dir = DRIVE_ROOT/"logs"
    log_dir.mkdir(exist_ok=True)
    LOG_PATH = log_dir/("train_"+time.strftime("%Y%m%d_%H%M%S")+".log")
    from colab_video_runtime import run_foreground
    print("Resuming phase", state["step"], "| target:", "continuous" if TARGET_PHASE_STEP == 0 else TARGET_PHASE_STEP)
    print("Leave this cell running. Press stop once for a clean save; wait for STOPPED phase.")
    run_foreground(command, LAB, LOG_PATH, STOP_FILE)


## 5. Saved status
The Start cell displays live progress. Use this cell after stopping to inspect the last saved metrics.


In [ ]:
print("Training active:", worker_active())
if "TRAIN_PROCESS" in globals():
    print("Process exit code:", TRAIN_PROCESS.poll(), "(None means still running)")
if "LOG_PATH" in globals() and LOG_PATH.exists():
    with LOG_PATH.open("rb") as f:
        f.seek(max(0, LOG_PATH.stat().st_size-10000))
        print(f.read().decode("utf-8", errors="replace"))
metrics = json.loads((RUN/"metrics_latest.json").read_text())
print(json.dumps(metrics, indent=2))
print("Saved phase step:", metrics["phase_step"], "| target:", TARGET_PHASE_STEP)



## 6. Stop and save — optional
Set the flag to `True` and run this cell. The trainer finishes its current update, evaluates, writes the checkpoint, verifies the Drive copy, and exits. Wait for **Training active: False** before deleting or disconnecting the runtime. An unexpected runtime loss can discard updates after the last completed Drive snapshot.

For the default foreground run, press the Start cell stop button once and wait for the clean-save message.


In [ ]:
REQUEST_STOP = False  # Change to True only when you want to stop.
if REQUEST_STOP:
    STOP_FILE.touch()
    print("Stop/save requested. Rerun Status until the trainer exits and Drive verification appears.")
else:
    print("No stop requested.")



## 7. Render the latest JC video — run after training stops
Rendering uses a CPU copy to preserve the original Blender geometry and exact registered keyframes. It does not consume optimizer steps. Preview: 72 frames, 24 fps, 960×540. Eight source keyframes must pass the zero-change check before the video is copied to Drive.


In [ ]:
from IPython.display import display, Video
from types import SimpleNamespace
if worker_active():
    print("Training is active. Run this cell again after it finishes or after Stop/save.")
else:
    from v07_preview import export_preview
    torch.set_num_threads(4)
    checkpoint = torch.load(RUN/"latest.pt", map_location="cpu", weights_only=True)
    model = PoseInterpolator(checkpoint["config"]["size"], checkpoint["config"]["width"])
    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval()
    jc = PoseData(DEFAULT_ASSETS, checkpoint["config"]["size"])
    jc.attach_shot(SHOT)
    preview = LOCAL/("preview_phase_"+str(checkpoint["step"])+"_"+uuid.uuid4().hex[:6])
    preview.mkdir()
    args = SimpleNamespace(out=str(preview),frames=72,fps=24,render_width=960,render_height=540)
    export_preview(model,jc,args,checkpoint["step"])
    manifest = json.loads((preview/"preview_manifest.json").read_text())
    assert manifest["keyframe_count"] == 8
    assert manifest["max_registered_keyframe_error"] == 0
    assert manifest["frames"] == 72
    probe = json.loads(subprocess.check_output(["ffprobe","-v","error","-count_frames",
        "-select_streams","v:0","-show_entries","stream=nb_read_frames,width,height",
        "-of","json",str(preview/"preview_latest.mp4")]))["streams"][0]
    assert int(probe["nb_read_frames"]) == 72 and probe["width"] == 960 and probe["height"] == 540
    print("Verified 72 frames and 8 unchanged registered keys")
    destination = DRIVE_ROOT/"previews"/preview.name
    destination.mkdir(parents=True)
    for path in preview.iterdir():
        if path.is_file():
            shutil.copyfile(path,destination/path.name)
            assert digest(path) == digest(destination/path.name)
    display(Video(str(preview/"preview_latest.mp4"), embed=True))
    print("Preview and review files:",destination)



## 8. Training journal
The verified pre-Colab experiment is recorded below. Lower L1 is better.

| Measurement | Before open-data fine-tuning | Saved phase100 |
|---|---:|---:|
| Held-out Sintel model RGB L1 | 0.034161 | 0.033196 |
| Held-out crossfade RGB L1 | 0.027657 | 0.027657 |
| Held-out RAFT-only warp RGB L1 | 0.032425 | 0.032425 |
| Original JC pose model L1 | 0.012097 | 0.011709 |

Best held-out checkpoint was phase75, L1 **0.032948**. It still trails crossfade.
The held-out film helped choose a checkpoint, so it is a **validation set**, not a sealed final test.
More steps on 57 training triplets can overfit; a larger independent test set and visual review are needed before claiming general animation quality.

Authoring validation: **27 tests passed, 1 CUDA test skipped** in the CPU-only workspace; actual CPU checkpoint resume 100→101, clean stop, snapshot roundtrip, corruption fallback, and rollback protection passed. This smoke test used an isolated copy; the supplied bundle remains at phase100. CUDA and real Drive execution await this notebook's Colab run.

The following cell writes a current journal to Drive. It records the saved checkpoint, runtime and actual metric values.


In [ ]:
from IPython.display import Markdown
metrics = json.loads((RUN/"metrics_latest.json").read_text())
journal = {
    "recorded_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "code_commit": CODE_COMMIT, "bundle_sha256": BUNDLE_SHA256,
    "gpu": torch.cuda.get_device_name(0), "torch": str(torch.__version__),
    "cuda_smoke_test_passed": CUDA_TEST_PASSED, "training_active": worker_active(),
    "target_phase_step": TARGET_PHASE_STEP, "saved_metrics": metrics,
    "scope": "V0.8 RGBA interpolation with frozen RAFT and JC rehearsal; not Wan fine-tuning"
}
(DRIVE_ROOT/"training_journal_latest.json").write_text(json.dumps(journal,indent=2)+"\n")
text = (
    "# Quantonium Video training journal\n\n"
    f"Recorded UTC: {journal['recorded_utc']}\n\n"
    f"Code: {CODE_COMMIT}\n\n"
    f"GPU: {journal['gpu']}; PyTorch: {journal['torch']}\n\n"
    f"Saved phase step: {metrics['phase_step']}; target: {TARGET_PHASE_STEP}\n\n"
    f"Held-out movie model L1: {metrics['external']['model_l1']:.6f}\n\n"
    f"Crossfade L1: {metrics['external']['crossfade_l1']:.6f}\n\n"
    f"JC pose L1: {metrics['jc']['model_l1']:.6f}\n\n"
    "Status: experimental; inspect previews and independent validation before promotion.\n"
)
(DRIVE_ROOT/"training_journal_latest.md").write_text(text)
display(Markdown(text))



## 9. Wan generation training
Open the [Wan LoRA Colab notebook](https://colab.research.google.com/github/Anharmoniclabs/quantoniumoslinux/blob/feat/quantonium-anime-video-v01/runtime/experiments/quantonium_video/Quantonium_Wan_LoRA_Colab.ipynb). Run its sections 1–3, then leave its training cell running. It saves actual generator adapter updates and optimizer state every 25 updates, and loads the trained adapter for previews. Your V0.8 weights remain here. Stop this interpolator before using Wan on the same GPU.


In [ ]:
from IPython.display import Markdown, display
display(Markdown('[Open Wan generation training](https://colab.research.google.com/github/Anharmoniclabs/quantoniumoslinux/blob/feat/quantonium-anime-video-v01/runtime/experiments/quantonium_video/Quantonium_Wan_LoRA_Colab.ipynb)'))


### Resume after a disconnect
Reconnect a GPU runtime, rerun configuration/setup/restore, then Start. Keep target 0 to continue without a step cap. The most recent valid matching snapshot is restored; older snapshots and the initial bundle remain intact. If all completed snapshots fail validation, restoration stops rather than resetting training.

Snapshots accumulate intentionally (roughly a few MB each). They are not automatically deleted. Do not edit the original parent checkpoint, data manifest, or Blender shot. To use new data or change optimizer settings, create a separate fine-tuning lineage.

### Quality review before calling a run final
Check face/eye identity, pose anatomy, occlusion, outlines, foot contact and flicker in motion. Compare with the starting JC preview and held-out crossfade. Source keyframes being exact does not prove the learned in-betweens are accurate. This notebook gives a reproducible training handoff, not a claim of Seedance-level output.
